# Lab 4: Weather Data Pipeline

**Week 4 · Data Engineering Course**

---

## What You Will Build

A complete, end-to-end data pipeline that:

1. **Fetches** one month of historical weather data for 5 African cities from the Open-Meteo API
2. **Validates and cleans** the data
3. **Saves** it to `data/clean/weather_2024_jan.csv`
4. **Loads** it into a PostgreSQL table called `weather_daily`
5. **Answers** three business questions with SQL

Pipeline flow:
```
Open-Meteo API  →  pandas DataFrame  →  CSV  →  PostgreSQL
```

---

## Prerequisites

- Completed Lessons 1–4
- PostgreSQL running with a `weather_db` database
- `.env` file in this folder with your database password

```
# .env
DB_HOST=localhost
DB_PORT=5432
DB_NAME=weather_db
DB_USER=postgres
DB_PASSWORD=your_password_here
```

**Install packages:**
```
pip install requests pandas psycopg2-binary python-dotenv
```

---

## Step 1: Setup

In [ ]:
import os
import time
import json
import logging
import requests
import psycopg2
import pandas as pd
from pathlib import Path
from datetime import datetime

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DATA  = Path('../data')
CLEAN = DATA / 'clean'
LOGS  = Path('../logs')
for p in [CLEAN, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

# Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOGS / 'lab4.log', encoding='utf-8'),
    ]
)
log = logging.getLogger('lab4')

log.info('Lab 4 started')
print('Setup complete.')

---

## Step 2: Load Cities

In [ ]:
cities = pd.read_csv(DATA / 'cities.csv')
log.info(f'Loaded {len(cities)} cities')
print(cities)

---

## Step 3: Fetch Historical Weather

We use the Open-Meteo **archive** endpoint to get data for January 2024. Historical data is stable — it does not change each time you run the pipeline.

In [ ]:
ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'
START_DATE  = '2024-01-01'
END_DATE    = '2024-01-31'
FIELDS      = 'temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max'

def fetch_historical(city, lat, lon, tz):
    '''Fetch historical weather for one city. Returns a DataFrame or None.'''
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'start_date': START_DATE,
        'end_date':   END_DATE,
        'daily':      FIELDS,
        'timezone':   tz,
    }
    try:
        response = requests.get(ARCHIVE_URL, params=params, timeout=15)
        response.raise_for_status()
        raw = response.json()
    except requests.RequestException as e:
        log.error(f'Failed to fetch {city}: {e}')
        return None

    df = pd.DataFrame(raw['daily']).rename(columns={
        'time':                'date',
        'temperature_2m_max':  'temp_max_c',
        'temperature_2m_min':  'temp_min_c',
        'precipitation_sum':   'rain_mm',
        'windspeed_10m_max':   'wind_max_kmh',
    })
    df['date']      = pd.to_datetime(df['date'])
    df['city']      = city
    df['latitude']  = lat
    df['longitude'] = lon
    df['timezone']  = tz
    log.info(f'Fetched {len(df)} days for {city}')
    return df

print('fetch_historical() defined.')

In [ ]:
all_frames = []

for _, row in cities.iterrows():
    df = fetch_historical(row['city'], row['latitude'], row['longitude'], row['timezone'])
    if df is not None:
        all_frames.append(df)
    time.sleep(0.5)   # be polite to the API

if not all_frames:
    raise RuntimeError('No data fetched — check your internet connection')

weather_raw = pd.concat(all_frames, ignore_index=True)
log.info(f'Combined: {weather_raw.shape}')
print(f'\nRaw data: {weather_raw.shape}')
weather_raw.head()

---

## Step 4: Validate and Clean

In [ ]:
# Check for missing values
print('Missing values:')
print(weather_raw.isnull().sum())

In [ ]:
# Check row counts — each city should have 31 days
print('Rows per city:')
print(weather_raw.groupby('city').size().rename('days'))

In [ ]:
# Clean
weather = weather_raw.copy()

# Fill missing precipitation with 0 (no rain recorded = 0 mm)
weather['rain_mm'] = weather['rain_mm'].fillna(0.0)

# Round numerics
numeric_cols = ['temp_max_c', 'temp_min_c', 'rain_mm', 'wind_max_kmh']
weather[numeric_cols] = weather[numeric_cols].round(2)

# Reorder columns
weather = weather[['city', 'date', 'temp_max_c', 'temp_min_c', 'rain_mm',
                   'wind_max_kmh', 'latitude', 'longitude', 'timezone']]

# Final validation
assert weather.isnull().sum().sum() == 0, 'Still have missing values!'
assert (weather['rain_mm'] >= 0).all(), 'Negative precipitation found!'

log.info(f'Cleaned data: {weather.shape}, no nulls')
print(f'Clean data: {weather.shape}')
weather.describe()

---

## Step 5: Save to CSV

In [ ]:
csv_path = CLEAN / 'weather_2024_jan.csv'
weather.to_csv(csv_path, index=False)
log.info(f'Saved to {csv_path}')
print(f'Saved {len(weather)} rows to {csv_path}')

# Verify
check = pd.read_csv(csv_path)
print(f'Re-read: {check.shape}')
check.head()

---

## Step 6: Load into PostgreSQL

In [ ]:
# Connect using credentials from .env
DB_CONFIG = {
    'host':     os.environ.get('DB_HOST', 'localhost'),
    'port':     int(os.environ.get('DB_PORT', 5432)),
    'dbname':   os.environ.get('DB_NAME', 'weather_db'),
    'user':     os.environ.get('DB_USER', 'postgres'),
    'password': os.environ.get('DB_PASSWORD', ''),
}

try:
    conn = psycopg2.connect(**DB_CONFIG)
    log.info('Connected to PostgreSQL')
    print('Connected.')
except psycopg2.OperationalError as e:
    log.error(f'Connection failed: {e}')
    raise

In [ ]:
# Create the table
create_sql = '''
    CREATE TABLE IF NOT EXISTS weather_daily (
        id           SERIAL PRIMARY KEY,
        city         VARCHAR(100) NOT NULL,
        date         DATE         NOT NULL,
        temp_max_c   NUMERIC(5,2),
        temp_min_c   NUMERIC(5,2),
        rain_mm      NUMERIC(7,2),
        wind_max_kmh NUMERIC(6,2),
        latitude     NUMERIC(8,4),
        longitude    NUMERIC(8,4),
        timezone     VARCHAR(50),
        UNIQUE (city, date)
    );
'''

with conn.cursor() as cur:
    cur.execute(create_sql)
conn.commit()
log.info('Table weather_daily ready')
print('Table ready.')

In [ ]:
# Upsert all rows
upsert_sql = '''
    INSERT INTO weather_daily
        (city, date, temp_max_c, temp_min_c, rain_mm, wind_max_kmh,
         latitude, longitude, timezone)
    VALUES
        (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (city, date) DO UPDATE SET
        temp_max_c   = EXCLUDED.temp_max_c,
        temp_min_c   = EXCLUDED.temp_min_c,
        rain_mm      = EXCLUDED.rain_mm,
        wind_max_kmh = EXCLUDED.wind_max_kmh;
'''

cols = ['city', 'date', 'temp_max_c', 'temp_min_c', 'rain_mm',
        'wind_max_kmh', 'latitude', 'longitude', 'timezone']

rows = [tuple(r) for r in weather[cols].itertuples(index=False)]

with conn.cursor() as cur:
    cur.executemany(upsert_sql, rows)
conn.commit()

log.info(f'Upserted {len(rows)} rows into weather_daily')
print(f'Upserted {len(rows)} rows.')

In [ ]:
# Verify the row count in the database
with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM weather_daily;')
    count = cur.fetchone()[0]

print(f'Rows in weather_daily: {count}')
assert count == len(weather), f'Expected {len(weather)} rows, got {count}'

---

## Step 7: Answer Business Questions

In [ ]:
# Question 1: Which city had the highest average maximum temperature in January 2024?
q1 = pd.read_sql('''
    SELECT
        city,
        ROUND(AVG(temp_max_c), 1) AS avg_max_temp_c
    FROM weather_daily
    GROUP BY city
    ORDER BY avg_max_temp_c DESC;
''', conn)

print('Q1: Average maximum temperature by city (January 2024)')
print(q1.to_string(index=False))

In [ ]:
# Question 2: Which city received the most rainfall? How many rainy days did it have?
q2 = pd.read_sql('''
    SELECT
        city,
        ROUND(SUM(rain_mm), 1)          AS total_rain_mm,
        COUNT(*) FILTER (WHERE rain_mm > 0) AS rainy_days
    FROM weather_daily
    GROUP BY city
    ORDER BY total_rain_mm DESC;
''', conn)

print('Q2: Total rainfall and rainy days by city')
print(q2.to_string(index=False))

In [ ]:
# Question 3: What were the top 5 hottest days across all cities?
q3 = pd.read_sql('''
    SELECT
        city,
        date,
        temp_max_c
    FROM weather_daily
    ORDER BY temp_max_c DESC
    LIMIT 5;
''', conn)

print('Q3: Top 5 hottest days in January 2024')
print(q3.to_string(index=False))

In [ ]:
conn.close()
log.info('Lab 4 complete.')
print('\nDone! Check logs/lab4.log for the full run log.')

---

## Checklist

- [ ] `data/clean/weather_2024_jan.csv` exists and has 155 rows (5 cities × 31 days)
- [ ] No null values in any column
- [ ] `weather_daily` table has 155 rows in PostgreSQL
- [ ] Re-running the pipeline does not create duplicate rows (upsert works)
- [ ] All three business questions are answered
- [ ] `logs/lab4.log` exists and shows the full pipeline run
- [ ] Notebook runs from top to bottom with no errors